# LogHAR Model on the MHAR Feature Set


**Purpose:** The purpose of this notebook is to compute the out-of-sample forecast performance of the LogHAR model on the $\mathcal{M}_{\mathrm{HAR}}$ feature set for the EURO STOXX 50 index. Following the same benchmark workflow as the HAR notebook, we estimate the model on a rolling window and evaluate one-day-ahead forecasts using the Mean Squared Error (MSE). In this specification, both the dependent variable and predictors remain in log space: the target is `logRV_target`, and the predictors are `logRVD`, `logRVW`, and `logRVM`.


## 1.1 Loading Data


In [ ]:
import pandas as pd
import numpy as np

# Load MALL dataset
file_path = "/Users/tobiasbergdahlpersson/Documents/SSE/MSc Thesis/Cleaned Data/MALL.csv"
MALL = pd.read_csv(file_path, index_col='Date', parse_dates=True)

print(f"Shape: {MALL.shape}")
print(f"Date range: {MALL.index[0].date()} to {MALL.index[-1].date()}")
print(f"NaN values: {MALL.isna().sum().sum()}")
print("\nLogHAR columns available:")
print(MALL[["logRVD", "logRVW", "logRVM", "logRV_target"]].head())


## 2. Data Split


Following Christensen et al. (2023), we split the sample into a training set (70%), validation set (10%), and test set (20%) in chronological order. The rolling estimation window used for out-of-sample forecasting is the combined training and validation sample, matching the HAR benchmark notebook.


In [ ]:
# Train / Validation / Test split (70 / 10 / 20)
n = len(MALL)
n_train = int(np.floor(0.70 * n))
n_val   = int(np.floor(0.10 * n))
n_test  = n - n_train - n_val

train_idx = MALL.index[:n_train]
val_idx   = MALL.index[n_train:n_train + n_val]
test_idx  = MALL.index[n_train + n_val:]

print(f"Total observations: {n}")
print(f"\nTrain:      {train_idx[0].date()} to {train_idx[-1].date()} ({n_train} obs, {n_train/n*100:.1f}%)")
print(f"Validation: {val_idx[0].date()} to {val_idx[-1].date()} ({n_val} obs, {n_val/n*100:.1f}%)")
print(f"Test:       {test_idx[0].date()} to {test_idx[-1].date()} ({n_test} obs, {n_test/n*100:.1f}%)")


## 3. LogHAR Model


The LogHAR benchmark is estimated in log space as

$$\log(RV_{t+1}) = \beta_0 + \beta_d \log(RV_t) + \beta_w \log(\overline{RV}^{(w)}_t) + \beta_m \log(\overline{RV}^{(m)}_t) + \varepsilon_{t+1}$$

where the target is `logRV_target` and the predictors are the log-transformed MHAR variables available in `MALL.csv`.


### 3.2 Feature Set Construction


In [ ]:
# Define target and feature set for LogHAR on MHAR
y = MALL["logRV_target"]
X_MHAR = MALL[["logRVD", "logRVW", "logRVM"]]

print("Target variable: logRV_target")
print(f"Predictors: {X_MHAR.columns.tolist()}")
print(f"\nFirst few rows:")
print(pd.concat([y, X_MHAR], axis=1).head())


### 3.3 Rolling Window Estimation and Forecasting


In [ ]:
from sklearn.linear_model import LinearRegression

# Rolling window length = train + validation
window = n_train + n_val

forecasts_LogHAR = []
dates_test = []

for i in range(n_test):
    # Extract rolling window
    X_window = X_MHAR.iloc[i : i + window].values
    y_window = y.iloc[i : i + window].values

    # OLS estimation in log space
    model = LinearRegression()
    model.fit(X_window, y_window)

    # One-step-ahead forecast using time t predictors
    X_forecast = X_MHAR.iloc[i + window].values.reshape(1, -1)
    forecast = model.predict(X_forecast)[0]

    forecasts_LogHAR.append(forecast)
    dates_test.append(MALL.index[i + window])

print(f"Forecasts generated: {len(forecasts_LogHAR)}")
print(f"Forecast period: {dates_test[0].date()} to {dates_test[-1].date()}")


### 3.4 Out-of-Sample Performance Evaluation


Following Christensen et al. (2023), out-of-sample forecast accuracy is evaluated using the Mean Squared Error (MSE). For the LogHAR notebook, forecasts and actual values are both compared in log space.


In [ ]:
# Collect results
results_LogHAR = pd.DataFrame({
    "logRV_actual": y.loc[dates_test].values,
    "logRV_forecast": forecasts_LogHAR
}, index=dates_test)

# Compute out-of-sample MSE
mse_LogHAR = np.mean((results_LogHAR["logRV_actual"] - results_LogHAR["logRV_forecast"])**2)

print(f"Out-of-sample MSE (LogHAR): {mse_LogHAR:.6e}")
print(f"\nFirst few forecasts vs actuals:")
print(results_LogHAR.head(10).to_string())


### 3.4.1 Save Forecasts


In [ ]:
import os

# Save LogHAR forecasts to CSV for comparison with other models
forecast_path = "/Users/tobiasbergdahlpersson/Documents/SSE/MSc Thesis/Cleaned Data/Forcasts/MHAR"
os.makedirs(forecast_path, exist_ok=True)

# Save forecasts
results_LogHAR.to_csv(os.path.join(forecast_path, "forecasts_LogHAR.csv"))

# Save MSE summary
mse_summary_LogHAR = pd.DataFrame({
    "Model": ["LogHAR"],
    "MSE":   [mse_LogHAR],
    "RMSE":  [np.sqrt(mse_LogHAR)],
    "MAE":   [np.mean(np.abs(results_LogHAR["logRV_actual"] - results_LogHAR["logRV_forecast"]))],
    "Mean_Error": [np.mean(results_LogHAR["logRV_actual"] - results_LogHAR["logRV_forecast"])],
}).set_index("Model")

mse_summary_LogHAR.to_csv(os.path.join(forecast_path, "mse_summary_LogHAR.csv"))

print(f"Forecasts saved to: {forecast_path}")
print(f"Files saved:")
print(f"  - forecasts_LogHAR.csv ({len(results_LogHAR)} rows)")
print(f"  - mse_summary_LogHAR.csv")
print(f"\nLogHAR MSE: {mse_LogHAR:.6e}")


## 4. Results


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(results_LogHAR["logRV_actual"], label="Actual log RV", color="black", linewidth=0.8)
ax.plot(results_LogHAR["logRV_forecast"], label="LogHAR Forecast", color="red", linewidth=0.8, linestyle="--")
ax.set_title("LogHAR Model - One-Day-Ahead Log Realized Variance Forecast\nEURO STOXX 50 (2016-2020)", fontsize=12)
ax.set_ylabel("Log Realized Variance")
ax.set_xlabel("Date")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nSummary of forecast errors:")
errors = results_LogHAR["logRV_actual"] - results_LogHAR["logRV_forecast"]
print(f"Mean error:  {errors.mean():.6e}")
print(f"RMSE:        {np.sqrt(mse_LogHAR):.6e}")
print(f"MAE:         {errors.abs().mean():.6e}")


### 4.2 Forecast Performance Summary


In [ ]:
from IPython.display import display

# Summary table of forecast performance
performance = pd.DataFrame({
    "Model": ["LogHAR"],
    "MSE":   [mse_LogHAR],
    "RMSE":  [np.sqrt(mse_LogHAR)],
    "MAE":   [errors.abs().mean()],
    "Mean Error": [errors.mean()],
    "Relative MSE": [1.0]
}).set_index("Model")

styled_performance = (
    performance.style
    .set_caption(
        "Table 3: Out-of-Sample Forecast Performance - LogHAR Model\n"
        f"EURO STOXX 50 Test Period: {dates_test[0].date()} to {dates_test[-1].date()} ({n_test} observations)"
    )
    .format({
        "MSE":         "{:.6e}",
        "RMSE":        "{:.6e}",
        "MAE":         "{:.6e}",
        "Mean Error":  "{:.6e}",
        "Relative MSE": "{:.4f}"
    })
    .set_table_styles([
        {"selector": "caption",
         "props": [
             ("font-size", "12px"),
             ("font-weight", "bold"),
             ("text-align", "left"),
             ("padding-bottom", "8px"),
             ("color", "black"),
             ("caption-side", "top"),
         ]},
        {"selector": "thead tr th",
         "props": [
             ("background-color", "white"),
             ("color", "black"),
             ("font-size", "11px"),
             ("text-align", "center"),
             ("padding", "6px 10px"),
             ("border-top", "2px solid black"),
             ("border-bottom", "1px solid black"),
             ("font-weight", "bold"),
         ]},
        {"selector": "tbody tr th",
         "props": [
             ("font-size", "11px"),
             ("font-weight", "bold"),
             ("text-align", "left"),
             ("padding", "4px 10px"),
             ("color", "black"),
             ("background-color", "white"),
             ("border-right", "1px solid black"),
         ]},
        {"selector": "tbody tr td",
         "props": [
             ("font-size", "11px"),
             ("text-align", "right"),
             ("padding", "4px 10px"),
             ("background-color", "white"),
             ("color", "black"),
         ]},
        {"selector": "tbody tr:last-child td, tbody tr:last-child th",
         "props": [("border-bottom", "2px solid black")]},
        {"selector": "table",
         "props": [
             ("border-collapse", "collapse"),
             ("width", "100%"),
         ]},
        {"selector": "td, th",
         "props": [
             ("border-left", "none"),
             ("border-right", "none"),
         ]},
    ])
)

display(styled_performance)
